# Block encoding: from a Hamiltonian to a polynomial

A block encoding puts a matrix inside a larger **unitary quantum operation**. This lets us use a Hamiltonian in algorithms even though the Hamiltonian itself is not a quantum gate.

This demo introduces the library's **LCU, controlled LCU, qubitisation, QSVT and tensor hypercontraction (THC)** routines, using [guppy](https://docs.quantinuum.com/guppy/language_guide/language_guide_index.html). Run it from a source checkout with the development dependencies installed; the numerical checks use the repository's `tests.helpers`. The repository default is little endian.

For an ancilla register of $a$ qubits, our starting point is

$$
U=\begin{pmatrix}H/\lambda & *\\ * & *\end{pmatrix},
\qquad
(\langle 0^a|\otimes I)U(|0^a\rangle\otimes I)=H/\lambda.
$$

- **Build:** turn a weighted sum of Pauli operators into an LCU block encoding.
- **Compose:** add a control, form a quantum walk, or transform the encoded matrix with QSVT.
- **Check:** extract the small encoded blocks and compare them with the equations below.

In [1]:
import numpy as np
import zixy.qubit.pauli as zqp
from IPython.display import Math, display
from guppylang import guppy
from guppylang.std.builtins import array, comptime, dagger
from guppylang.std.quantum import h, qubit

from guppyalgos.algorithms.block_encoding.lcu import (
    LCU, LCUCntrl, LCUData,
    build_single_cntrl_select, build_cntrl_single_cntrl_select,
    build_double_cntrl_select, build_unary_iteration_select,
    build_cntrl_unary_iteration_select,
)
from guppyalgos.algorithms.block_encoding.qubitization import (
    Qubitization, QubitizationCntrl,
)
from guppyalgos.algorithms.block_encoding.qsvt import QSVT
from guppyalgos.algorithms.state_preparation import multiplexor_prep
from guppyalgos.primitives.gate_decompositions.cnx.cnx import cnx
from guppyalgos.primitives.subroutines.reflection import Reflection, ReflectionCntrl
from tests.helpers import Endianness, get_unitary_projected, assert_allclose_ignorephase


def show_block(label, matrix):
    # Compact LaTeX output, suitable for both Jupyter and the documentation.
    matrix = np.real_if_close(matrix)
    rows = [' & '.join(f'{v:.3f}' for v in row) for row in matrix]
    display(Math(label + r'=\begin{pmatrix}' + r'\\'.join(rows) + r'\end{pmatrix}'))


def check_block(actual, expected):
    assert_allclose_ignorephase(actual, expected, threshold=1e-7)
    # Align only the simulator's physically irrelevant global phase for display.
    overlap = np.vdot(expected, actual)
    return actual * np.exp(-1j * np.angle(overlap))


def block(circuit, post=None, pre=None):
    return get_unitary_projected(
        circuit, 1, post or {"prep": [False, False]},
        pre_select_dict=pre, endianness=Endianness.LITTLE,
        n_extra_qubits=4,
    )

## 1. Encode a weighted sum with LCU

Choose a one-qubit Hamiltonian whose answer fits on the page:

$$
H=0.5I+0.3Z+0.2X,\qquad
\lambda=\sum_j|\alpha_j|=1,\qquad
A=H/\lambda=\begin{pmatrix}0.8&0.2\\0.2&0.2\end{pmatrix}.
$$

`LCUData` computes the normalization and PREPARE amplitudes. Two ancilla qubits address the three terms; the unused address has zero amplitude.

$$
P|00\rangle=\sum_j\sqrt{|\alpha_j|/\lambda}\,|j\rangle,
\qquad
S=\sum_j|j\rangle\langle j|\otimes\operatorname{sgn}(\alpha_j)P_j,
\qquad U=P^\dagger SP.
$$

`LCU` composes **PREPARE → SELECT → UNPREPARE**. The SELECT builder includes the coefficient signs (or phases for complex coefficients).

In [2]:
hamiltonian = zqp.RealTermSum.from_str("(0.5, I0), (0.3, Z0), (0.2, X0)", 1)
data = LCUData.from_hamiltonian(hamiltonian)
A = hamiltonian.to_sparse_matrix().toarray() / data.l1_norm
prepare = multiplexor_prep(data.amplitudes)
select = build_double_cntrl_select(data)


@guppy
def unprepare(prep_qreg: array[qubit, 2]) -> None:
    with dagger:
        prepare(prep_qreg)


@guppy
def encode(prep_qreg: array[qubit, 2], qreg: array[qubit, 1]) -> None:
    LCU(prepare, select, unprepare).compose(prep_qreg, qreg)


encoded = block(encode)
encoded = check_block(encoded, A)
show_block(r'\langle 00|U|00\rangle', encoded)

<IPython.core.display.Math object>

The check extracts an **unnormalized block**, not a unitary acting on the system alone. On input $|\psi\rangle$, measuring the ancillas as $00$ succeeds with probability $\|A|\psi\rangle\|^2$ and leaves the normalized state $A|\psi\rangle/\|A|\psi\rangle\|$.

### Choose a SELECT implementation

The composition stays the same when you change how the address selects a term.

| Builder | Use it for |
| :-- | :-- |
| `build_single_cntrl_select` | Two terms, addressed by one ancilla. |
| `build_double_cntrl_select` | Three or four terms, addressed by two ancillas. |
| `build_unary_iteration_select` | Larger sums; shares address-decoding work across terms. Also supports the small cases. |
| `build_cntrl_single_cntrl_select` | A two-term SELECT with an additional external control. |
| `build_cntrl_unary_iteration_select` | A unary-iteration SELECT with an additional external control. |

Here “single” and “double” describe the **address controls**. The `cntrl` prefix adds a separate control for the whole operation. Unary iteration lets you choose compute/uncompute AND routines to tune its implementation.

In [3]:
unary_select = build_unary_iteration_select(data)


@guppy
def encode_unary(prep_qreg: array[qubit, 2], qreg: array[qubit, 1]) -> None:
    LCU(prepare, unary_select, unprepare).compose(prep_qreg, qreg)


check_block(block(encode_unary), A)

# The specialized one-address-qubit builders take a two-term sum.
two_terms = LCUData.from_hamiltonian(
    zqp.RealTermSum.from_str("(0.7, Z0), (0.3, X0)", 1)
)
small_select = build_single_cntrl_select(two_terms)
small_controlled_select = build_cntrl_single_cntrl_select(two_terms)
small_select.compile_function()
small_controlled_select.compile_function()
print("Both SELECT implementations encode A; the two-term builders compile.")

Both SELECT implementations encode A; the two-term builders compile.


## 2. Add a coherent control

`LCUCntrl` controls SELECT while reusing the same PREPARE and UNPREPARE. When the control is zero, they cancel; when it is one, the block encoding runs:

$$
\operatorname{ctrl}(U)=|0\rangle\langle0|\otimes I+|1\rangle\langle1|\otimes U.
$$

This is the building block needed for controlled walks and phase estimation. To check the control coherently, we surround it with Hadamards: starting at zero, the two control outcomes encode $(I+A)/2$ and $(I-A)/2$.

In [4]:
controlled_select = build_cntrl_unary_iteration_select(data)


@guppy
def controlled_encode(
    control_qreg: array[qubit, 1], prep_qreg: array[qubit, 2], qreg: array[qubit, 1]
) -> None:
    h(control_qreg[0])
    LCUCntrl(prepare, controlled_select, unprepare).compose(
        control_qreg[0], prep_qreg, qreg
    )
    h(control_qreg[0])


for enabled, expected in [(False, (np.eye(2) + A) / 2), (True, (np.eye(2) - A) / 2)]:
    projection = {"control": [enabled], "prep": [False, False]}
    actual = block(controlled_encode, projection)
    check_block(actual, expected)
print("Interference check: the control outcomes encode (I + A)/2 and (I - A)/2.")

Interference check: the control outcomes encode (I + A)/2 and (I - A)/2.


## 3. Turn the encoding into a quantum walk

For this Hermitian Pauli LCU, `Qubitization` adds the library's all-zero reflection:

$$
R=I-2|00\rangle\langle00|,\qquad W=RU,
\qquad \langle00|W^d|00\rangle=(-1)^dT_d(A).
$$

The walk converts eigenvalues into phases, making them accessible to phase estimation. For an eigenvalue $E$ of $H$, its walk eigenphases obey $\cos\theta=-E/\lambda$. Powers produce Chebyshev polynomials; for example $T_2(A)=2A^2-I$.

Use `QubitizationCntrl` when a phase-estimation control must enable the walk. Its reflection must be controlled too.

In [5]:
@guppy
def walk_squared(prep_qreg: array[qubit, 2], qreg: array[qubit, 1]) -> None:
    Qubitization(
        LCU(prepare, select, unprepare), Reflection[2, 1](cnx)
    ).power(prep_qreg, qreg, 2)


@guppy
def controlled_walk(
    control_qreg: array[qubit, 1], prep_qreg: array[qubit, 2], qreg: array[qubit, 1]
) -> None:
    h(control_qreg[0])
    QubitizationCntrl(
        LCUCntrl(prepare, controlled_select, unprepare), ReflectionCntrl[2](cnx)
    ).power(control_qreg[0], prep_qreg, qreg, 1)
    h(control_qreg[0])


T2 = 2 * A @ A - np.eye(2)
walk_block = check_block(block(walk_squared), T2)
for enabled, expected in [(False, (np.eye(2) - A) / 2), (True, (np.eye(2) + A) / 2)]:
    projection = {"control": [enabled], "prep": [False, False]}
    check_block(block(controlled_walk, projection), expected)
show_block(r'T_2(A)', walk_block)

<IPython.core.display.Math object>

## 4. Transform the block with QSVT

`QSVT` alternates a block encoding and its adjoint with phase rotations. The phases choose a polynomial transformation of the singular values. For our Hermitian example, we can show a simple exact polynomial:

$$
p(x)=1-2x^2=-T_2(x),\qquad
\langle0|_{s}\langle00|_{a}V|0\rangle_s|00\rangle_a=p(A).
$$

The two reflection-convention phases below are in **half-turns**. Here $U^\dagger=U$, so we may supply the same LCU for both arguments. A general encoding requires its actual adjoint. For designed approximations, `ChebyshevPolynomial` and `QSPAngleFinder` supply the polynomial and phase-generation workflow illustrated in the full QSVT example.

In [6]:
phases = [1.0, 1.0]


@guppy
def transform(
    prep_qreg: array[qubit, 2], signal_qreg: array[qubit, 1], qreg: array[qubit, 1]
) -> None:
    QSVT(
        LCU(prepare, select, unprepare),
        LCU(prepare, select, unprepare),
        comptime(phases),
    ).compose(signal_qreg[0], prep_qreg, qreg)


transformed = block(transform, {"prep": [False, False], "signal": [False]})
transformed = check_block(transformed, -T2)
show_block(r'p(A)', transformed)

<IPython.core.display.Math object>

## 5. Use structured chemistry data with THC

Tensor hypercontraction offers a structured alternative to enumerating a chemistry Hamiltonian as Pauli strings. The library separates **classical preprocessing** from the quantum routines that consume its tables:

- `build_thc_alias_terms` collects weighted terms. Alias sampling prepares their normalized weights, $p_j=|c_j|/\sum_k|c_k|$.
- `build_thc_lcu_data` builds the SELECT-record and Givens-angle QROMs, at chosen precision.
- `load_select_registers` loads the term indices and flags; `SelectTHCCntrl` combines the controlled selection with orbital rotations.
- `SelectTHCCntrlRegs` and `THCWalkTargetRegs` group the workspace for composition with `LCUCntrl` and `QubitizationCntrl`.

This small example builds the data and QROM callables. It does **not** construct a complete THC quantum encoding: that also needs alias PREPARE, rotation/resource-state preparation and workspace cleanup. The THC phase-estimation notebook links those pieces together.

In [7]:
from guppyalgos.algorithms.block_encoding.thc import (
    build_thc_alias_terms, build_thc_lcu_data, generate_thc_parameters,
)

# Synthetic data for a demo, rather than molecular integrals.
parameters = generate_thc_parameters(n_orbitals=2, thc_rank=2, seed=7)
terms = build_thc_alias_terms(parameters)
thc_data = build_thc_lcu_data(
    parameters, rotation_precision_bits=4, alias_precision_bits=4
)
weights = np.array([abs(term.coefficient) for term in terms])
np.testing.assert_allclose(thc_data.alias_probabilities, weights / weights.sum())
print(f"{len(terms)} THC terms; {thc_data.n_alias_qubits} alias-address qubits.")

5 THC terms; 3 alias-address qubits.


## Explore a complete workflow

- **LCU and walk powers:** [LCU and qubitisation](https://github.com/Quantinuum/guppy-algorithms/blob/main/examples/block_encoding/lcu_qubitization.ipynb).
- **Approximate a chosen function:** [QSVT](https://github.com/Quantinuum/guppy-algorithms/blob/main/examples/block_encoding/qsvt.ipynb) and [QSP angle generation](https://github.com/Quantinuum/guppy-algorithms/blob/main/examples/block_encoding/qsp_angle_finder.ipynb).
- **Assemble the chemistry components:** [THC phase estimation](https://github.com/Quantinuum/guppy-algorithms/blob/main/examples/phase_estimation/thc_qpe.ipynb).

The numerical checks preserve the block's scale while allowing an overall global phase. Interference on the external control checks the relative sign of the controlled walk. The displayed matrices are aligned to the equations' global-phase convention.